In [6]:
!pip install datasets

In [7]:
!pip install soundfile
!pip install librosa

^C
ERROR: Operation cancelled by user


In [8]:
!pip install PyFLAC

In [12]:
import soundfile as sf
from tqdm import tqdm

In [13]:
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load the datasets from sessions 1 to 5
all_datasets = []
# for session_id in range(1, 6):
    # dataset = load_dataset("confit/iemocap", f"session{session_id}")
    # dataset = load_dataset("Zahra99/IEMOCAP_Audio48")
    # dataset = load_dataset("Ar4ikov/iemocap_audio_text")
    # all_datasets.append(dataset)
dataset = load_dataset("ajyy/MELD_audio")
# dataset1 = load_dataset("declare-lab/MELD")


/opt/conda/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for ajyy/MELD_audio contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ajyy/MELD_audio
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/9988 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1108 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2610 [00:00<?, ? examples/s]

In [14]:
print(dataset1['train'][0])

NameError: name 'dataset1' is not defined

In [15]:
import pickle
if 1:
        (
            video_ids,
            video_speakers,
            video_labels,
            video_text,
            video_audio,
            video_visual,
            video_sentence,
            trainVids,
            test_vids,
            _,
        ) = pickle.load(
            # open("./data/iemocap/IEMOCAP_features.pkl", "rb"), encoding="latin1"
            open("/kaggle/input/melddd/MELD_features_final.pkl", "rb"), encoding="latin1" 
        )
# print(len(video_ids), type(video_ids), video_ids.keys())
print("Loaded pickle file!")

Loaded pickle file!


In [16]:
print(len(video_ids))
sum= 0
trainlen= 1132
for i in trainVids:
    sum+=len(video_ids[i])
print(sum)
sum=0    
for i in test_vids:
    sum+=len(video_ids[i])
print(sum)
# print(trainVids[0])

1432
11098
2610


In [17]:
print(dataset['train']['text'][1]==video_sentence[0][1])

True


In [18]:
import re
def extract_dialogue_utterance(path):
    # This regular expression will find the numbers in the format dia[number]_utt[number].flac
    match = re.search(r'dia(\d+)_utt(\d+)\.flac', path)
    if match:
        dialogue_num = int(match.group(1))
        utterance_num = int(match.group(2))
        return (dialogue_num, utterance_num)
    return (float('inf'), float('inf'))  # Return infinity if no numbers are found (handles edge cases)

In [19]:
print(dataset['validation']['text'][0]==video_sentence[1039][0])
print(dataset['test']['text'][0]==video_sentence[1153][0])

True
True


In [20]:
print((dataset['train']['text'][0:10]))
print(video_sentence[0])
print()
print(len(dataset['train']))
print(len(dataset['train'])+ len(dataset['validation']))
print
print(len(dataset['test']))
print(dataset['validation']['text'][0])
# print(dataset['train']['text'][-1], video_sentence[])

['also I was the point person on my company\x92s transition from the KL-5 to GR-6 system.', 'You must\x92ve had your hands full.', 'No don\x92t I beg of you!', 'All right then, we\x92ll have a definite answer for you on Monday, but I think I can say with some confidence, you\x92ll fit in well here.', 'Really?!', 'Absolutely.  You can relax', 'That I did. That I did.', 'So let\x92s talk a little bit about your duties.', 'My duties?  All right.', 'Now you\x92ll be heading a whole division, so you\x92ll have a lot of duties.']
['also I was the point person on my company\x92s transition from the KL-5 to GR-6 system.', 'You must\x92ve had your hands full.', 'That I did. That I did.', 'So let\x92s talk a little bit about your duties.', 'My duties?  All right.', 'Now you\x92ll be heading a whole division, so you\x92ll have a lot of duties.', 'I see.', 'But there\x92ll be perhaps 30 people under you so you can dump a certain amount on them.', 'Good to know.', 'We can go into detail', 'No don\x

In [21]:
# Initialize the Whisper processor and model for ASR
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

In [22]:
# Function to extract dialogue number
def extract_dialogue_number(path):
    match = re.search(r'dia(\d+)_utt\d+\.flac', path)
    if match:
        return int(match.group(1))
    return -1  # Return -1 if no dialogue number is found

def find_meld_samples(dataset, dia_num):
    relevant_samples = []
    flag= 0 # set to 1 if we miss an utterance.
    start= 0
    end= 0
    for i in range(len(trainVids)):
        if(extract_dialogue_number(dataset['train']['path'][start])!=i):
            print("Wrong start point")
        j= i+1
        while(extract_dialogue_number(dataset['train']['path'][j])==i):
            
        

SyntaxError: incomplete input (2790548016.py, line 19)

In [ ]:
print((dataset['train']['audio'][0].keys()))

In [ ]:
errors= [125, 556, 1149]
# Function to extract utterance number
def extract_number(utterance_path):
    match = re.search(r'utt(\d+)\.flac', utterance_path)
    if match:
        return int(match.group(1))
    return float('inf')  # Return infinity if no number is found (handles edge cases)

# Function to extract dialogue number
def extract_dialogue_number(path):
    match = re.search(r'dia(\d+)_utt\d+\.flac', path)
    if match:
        return int(match.group(1))
    return -1  # Return -1 if no dialogue number is found

# Function to find start and end indices of contiguous blocks with a particular dialogue number
def find_contiguous_indices(paths, dialogue_number):
    indices = [i for i, path in enumerate(paths) if extract_dialogue_number(path) == dialogue_number]
    
    if not indices:
        return None, None  # No indices found for the given dialogue number
    
    start_index = indices[0]
    end_index = indices[-1]
    return start_index, end_index

# Function to get sorted indices within a dialogue
def get_sorted_utterance_indices(paths, dialogue_number):
    start_idx, end_idx = find_contiguous_indices(paths, dialogue_number)
    if start_idx is None or end_idx is None:
        # print(f"Could not find dialogue number {dialogue_number}")
        return []
    # print(start_idx, end_idx)
    # Extract the utterances and their original indices within the dialogue
    utterances_with_indices = [(i, extract_number(paths[i])) for i in range(start_idx, end_idx + 1)]
    
    # Sort the utterances by their numbers
    sorted_utterances_with_indices = sorted(utterances_with_indices, key=lambda x: x[1])
    
    # Extract the sorted indices
    sorted_indices = [idx for idx, _ in sorted_utterances_with_indices]
    return sorted_indices

def sanity_check(dataset,name, sorted_indices, dialogue_number, video_sentence):
    flag= 0
    if(len(video_sentence[dialogue_number])!=len(sorted_indices)):
         print(f"NOT CORRECT, dia {dialogue_number}, name {name}, length of dia {len(video_sentence[dialogue_number])} {len(sorted_indices)} ")
         flag= 1
    else:
        for i in range(len(video_sentence[dialogue_number])):
            if(video_sentence[dialogue_number][i]!=dataset[name]['text'][sorted_indices[i]]):
                print(f"NOT CORRECT, dia {dialogue_number}, name {name}, length of dia {len(video_sentence[dialogue_number])} {len(sorted_indices)}, utterance number {i+1} is not same ")
                flag= 1
    if(flag):
        print(sorted_indices)
        print("Pickle:, ", video_sentence[dialogue_number])
        print("HF:, ")
        for num in sorted_indices:
             print(dataset[name]['text'][num])
            
def find_samples(dataset, trainVids, test_vids, video_sentence):
    trainlen= 1152 # to search till 1152 here. 
    samples= {}
    with open('/kaggle/working/samples.pkl', "rb") as fp:
        samples= pickle.load(fp)
#     pickle.dump(samples, '/kaggle/input/samples.pkl')
    for i in tqdm(trainVids, desc="Processing train videos"):
        if(i in errors):
            samples.append("Error")
            continue
        if(i in samples.keys()):
            continue
        dialogue_number_to_find = i
        name="train"
        sorted_indices = get_sorted_utterance_indices(dataset['train']['path'], dialogue_number_to_find)
        if(sorted_indices==[]):
            sorted_indices = get_sorted_utterance_indices(dataset['validation']['path'], dialogue_number_to_find-1039)
            name="validation"
        if(sorted_indices==[]):
            print(f"Not found {dialogue_number_to_find}")
#         sanity_check(dataset,name, sorted_indices, dialogue_number_to_find, video_sentence)
        temp= []
        for num in sorted_indices:
            temp.append(dataset[name]['audio'][num]) #/kaggle/input/melddd
            # samples.append(dataset[name]['text'][num]) 
        samples[i]= temp
        with open('/kaggle/working/samples.pkl', "wb") as fp:
            pickle.dump(samples, fp)
#         if(i==1):
#             return samples
    name="test"
    for i in tqdm(test_vids, desc="Processing test videos"):
        dialogue_number_to_find = i
        sorted_indices_test = get_sorted_utterance_indices(dataset['test']['path'], dialogue_number_to_find-1153)
        if(sorted_indices_test==[]):
            print(f"Not found {dialogue_number_to_find}")
#         sanity_check(dataset,name, sorted_indices_test, dialogue_number_to_find, video_sentence)
        temp=[]
        for num in sorted_indices_test:
            temp.append(dataset[name]['audio'][num])
                # samples.append(dataset[name]['text'][num]) 
        samples[i]= temp
        with open('/kaggle/working/samples.pkl', "wb") as fp:
            pickle.dump(samples, fp)
#         samples.append(temp)
    return samples
        
samples= find_samples(dataset, trainVids, test_vids, video_sentence)
     
# Example usage
dialogue_number_to_find = 1001
# print(dataset['train']['path'][18:30])
sorted_indices = get_sorted_utterance_indices(dataset['train']['path'], dialogue_number_to_find)
print(sorted_indices)

Processing train videos:   3%|▎         | 39/1152 [2:19:50<110:51:40, 358.58s/it]

In [29]:
print(len(samples[1]))

7


In [26]:
#There are three errors. 125, 556, and 1149: so I will just use the old text for them
for num in range(len(samples)):
    if(samples[num]=="Error"):
        print(num)
    # print(samples[num+1])

In [47]:
def generate_text_meld(samples): # returns the ASR text for the raw audio for each utterance in the dialogue(stored in samples)
    # Iterate over relevant samples to get ASR for each utterance
    video_sentences= {}
    for dia in tqdm(samples, desc="Iterating over dialogues"):
        temp= []
        for utterance in dia:
#             audio_sample = utterance[]
            waveform = utterance["array"]
            sampling_rate = utterance["sampling_rate"]
            # Use the model and processor to transcribe the audio:
            input_features = processor(
                waveform, sampling_rate=sampling_rate, return_tensors="pt"
            ).input_features
            # Generate token ids
            predicted_ids = model.generate(input_features)
            # Decode token ids to text
            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
            temp.append(transcription[0])
        video_sentences[dia]= temp
    return video_sentences

In [41]:
blah= generate_text_meld(samples)

Iterating over dialogues: 100%|██████████| 2/2 [00:16<00:00,  8.35s/it]


In [46]:
print(video_sentence[1])

['But then who? The waitress I went out with last month?', 'You know? Forget it!', 'No-no-no-no, no! Who, who were you talking about?', "No, I-I-I-I don't, I actually don't know", 'Ok!', 'All right, well...', 'Yeah, sure!']


In [45]:
print(blah[1])

[' The waitress I went out with last month?', ' Forget it.', " Forget it. No, no, no, who you talking about? No, I don't. I don't.", " I actually don't know who I'm talking about.", ' Okay.', ' Uh', ' Sure.']


In [ ]:
from tqdm import tqdm
new_sentences= {}
# Iterate over each video in video_ids
for video in tqdm(video_sentences.keys()):
    utterances_to_search = video_ids[video]
    samples, flag = find_samples(utterances_to_search, video)
    if flag == 0:
        text = generate_text(samples)
        new_sentences[video]= text
    else:
        print(f"Not generating ASR for {video}")

In [ ]:
def find_samples(utterances_to_search, name): # finds all the .wav samples(raw audio) corresponding to the current dialogue
    if(dataset=="iemocap"):
        utterances_to_search = [utterance + ".wav" for utterance in utterances_to_search]
    # Initialize a list to store relevant samples
    relevant_samples = []
    flag= 0 # set to 1 if we miss an utterance.
    # Iterate over each utterance to search for
    for utterance in utterances_to_search:
        # Flag to indicate if the utterance is found
        utterance_found = False
        # Iterate over each dataset in all_datasets
        for session in dataset.keys():
            # Iterate over the train split to find relevant samples
            for idx, sample in enumerate(dataset[session]):
                if utterance == sample['audio']['path']:
                    relevant_samples.append(sample)
                    # print(idx)
                    utterance_found = True
                    break  # Found the relevant sample, move to the next utterance
    if(len(relevant_samples)!=len(utterances_to_search)):
        print(f"Found only {len(relevant_samples)} utterances in a dialogue of length {len(utterances_to_search)} in dialogue {name}")
        # print("")
        flag= 1
    return relevant_samples, flag


In [ ]:
# print(text, len(text))

In [22]:
def generate_text(samples): # returns the ASR text for the raw audio for each utterance in the dialogue(stored in samples)
    # Iterate over relevant samples to get ASR for each utterance
    video_sentences= []
    for idx, sample in enumerate(samples, start=1):
        audio_sample = sample['audio']
        waveform = audio_sample["array"]
        sampling_rate = audio_sample["sampling_rate"]
        # Use the model and processor to transcribe the audio:
        input_features = processor(
            waveform, sampling_rate=sampling_rate, return_tensors="pt"
        ).input_features
        # Generate token ids
        predicted_ids = model.generate(input_features)
        # Decode token ids to text
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
        video_sentences.append(transcription[0])
    return video_sentences

In [26]:
from tqdm import tqdm

In [25]:
# print((trainVids))

In [26]:
# print(video_ids)

In [27]:
# for video in video_ids.keys():
#     print(video)

In [14]:
from tqdm import tqdm
new_sentences= {}
# Iterate over each video in video_ids
for video in tqdm(video_ids.keys(), desc="Processing videos"):
    utterances_to_search = video_ids[video]
    samples, flag = find_samples(utterances_to_search, video)
    if flag == 0:
        text = generate_text(samples)
        new_sentences[video]= text
    else:
        print(f"Not generating ASR for {video}")


Processing videos:   0%|          | 0/1432 [00:00<?, ?it/s]


TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
# Initialize an empty dictionary to store the mapping
sentences_map = {}

# Iterate over each element in new_sentences
for video, text in new_sentences:
    # Add the text as the key and the video as the value in the dictionary
    sentences_map[text] = video

# Now sentences_map is a dictionary where the text is the key

In [ ]:
# need to modify the sentences in this.
for video in video_sentence.keys():
      if(video_sentence[video]!= sentences_map[video]):
        video_sentence[video]= sentences_map[video]
      else:
        print(f"The ASR did not make any change in {video}")

In [ ]:
# Specify the path for the new pickle file
new_pickle_file_path = "/content/modified_IEMOCAP_features.pkl"

# Save the modified data to the new pickle file
with open(new_pickle_file_path, "wb") as f:
    pickle.dump(
        (
            video_ids,
            video_speakers,
            video_labels,
            video_text,
            video_audio,
            video_visual,
            video_sentence,
            trainVids,
            test_vids,
        ),
        f,
    )

print("Modified data saved to:", new_pickle_file_path)

In [39]:
print(video_sentence)

{0: ['also I was the point person on my company\x92s transition from the KL-5 to GR-6 system.', 'You must\x92ve had your hands full.', 'That I did. That I did.', 'So let\x92s talk a little bit about your duties.', 'My duties?  All right.', 'Now you\x92ll be heading a whole division, so you\x92ll have a lot of duties.', 'I see.', 'But there\x92ll be perhaps 30 people under you so you can dump a certain amount on them.', 'Good to know.', 'We can go into detail', 'No don\x92t I beg of you!', 'All right then, we\x92ll have a definite answer for you on Monday, but I think I can say with some confidence, you\x92ll fit in well here.', 'Really?!', 'Absolutely.  You can relax'], 1: ['But then who? The waitress I went out with last month?', 'You know? Forget it!', 'No-no-no-no, no! Who, who were you talking about?', "No, I-I-I-I don't, I actually don't know", 'Ok!', 'All right, well...', 'Yeah, sure!'], 2: ['Hey, Mon.', 'Hey-hey-hey. You wanna hear something that sucks.', 'Do I ever.', 'Chris sa

In [19]:
# List of utterances to search for in audio paths
# utterances_to_search = [
#     'Ses02F_impro05_F000.wav', 'Ses02F_impro05_M000.wav', 'Ses02F_impro05_F001.wav', 'Ses02F_impro05_M001.wav',
#     'Ses02F_impro05_M002.wav', 'Ses02F_impro05_M003.wav', 'Ses02F_impro05_M004.wav', 'Ses02F_impro05_F004.wav',
#     'Ses02F_impro05_M005.wav', 'Ses02F_impro05_M006.wav', 'Ses02F_impro05_F006.wav', 'Ses02F_impro05_M007.wav',
#     'Ses02F_impro05_F007.wav', 'Ses02F_impro05_M008.wav', 'Ses02F_impro05_M009.wav', 'Ses02F_impro05_F010.wav',
#     'Ses02F_impro05_M010.wav', 'Ses02F_impro05_F011.wav', 'Ses02F_impro05_M011.wav', 'Ses02F_impro05_M012.wav',
#     'Ses02F_impro05_F013.wav', 'Ses02F_impro05_M013.wav', 'Ses02F_impro05_F014.wav', 'Ses02F_impro05_M014.wav',
#     'Ses02F_impro05_F015.wav', 'Ses02F_impro05_M015.wav', 'Ses02F_impro05_F016.wav', 'Ses02F_impro05_M016.wav',
#     'Ses02F_impro05_F017.wav', 'Ses02F_impro05_M017.wav', 'Ses02F_impro05_M018.wav', 'Ses02F_impro05_F018.wav',
#     'Ses02F_impro05_M019.wav', 'Ses02F_impro05_F019.wav', 'Ses02F_impro05_M020.wav', 'Ses02F_impro05_F020.wav',
#     'Ses02F_impro05_M021.wav', 'Ses02F_impro05_F022.wav'
# ]
utterances_to_search= ['Ses02M_impro05_M000', 'Ses02M_impro05_F000', 'Ses02M_impro05_M001', 'Ses02M_impro05_F001', 'Ses02M_impro05_M002', 'Ses02M_impro05_F002', 'Ses02M_impro05_F003', 'Ses02M_impro05_M003', 'Ses02M_impro05_F004', 'Ses02M_impro05_F005', 'Ses02M_impro05_F006', 'Ses02M_impro05_F007', 'Ses02M_impro05_M006', 'Ses02M_impro05_F008', 'Ses02M_impro05_F009', 'Ses02M_impro05_M008', 'Ses02M_impro05_F010', 'Ses02M_impro05_F011', 'Ses02M_impro05_M010', 'Ses02M_impro05_M011', 'Ses02M_impro05_F012', 'Ses02M_impro05_F013', 'Ses02M_impro05_F014', 'Ses02M_impro05_M013', 'Ses02M_impro05_M014', 'Ses02M_impro05_F015', 'Ses02M_impro05_F016', 'Ses02M_impro05_M016', 'Ses02M_impro05_F017', 'Ses02M_impro05_F018', 'Ses02M_impro05_M017', 'Ses02M_impro05_M018', 'Ses02M_impro05_F019', 'Ses02M_impro05_M019', 'Ses02M_impro05_M020', 'Ses02M_impro05_F020', 'Ses02M_impro05_F021', 'Ses02M_impro05_M021', 'Ses02M_impro05_F022', 'Ses02M_impro05_F023', 'Ses02M_impro05_M023', 'Ses02M_impro05_M024', 'Ses02M_impro05_F024', 'Ses02M_impro05_M025', 'Ses02M_impro05_F025', 'Ses02M_impro05_F026', 'Ses02M_impro05_M026', 'Ses02M_impro05_M027', 'Ses02M_impro05_F028', 'Ses02M_impro05_M028', 'Ses02M_impro05_F029']
# Add ".wav" to each utterance dynamically
utterances_to_search = [utterance + ".wav" for utterance in utterances_to_search]

# Initialize a list to store relevant samples
relevant_samples = []

# Iterate over each utterance to search for
for utterance in utterances_to_search:
    # Flag to indicate if the utterance is found
    utterance_found = False
    # Iterate over each dataset in all_datasets
    for session in dataset.keys():
        # Iterate over the train split to find relevant samples
        for sample in dataset[session]:
            if utterance == sample['audio']['path']:
                relevant_samples.append(sample)
                utterance_found = True
                break  # Found the relevant sample, move to the next utterance
        # if not utterance_found:
        #     # Iterate over the test split to find relevant samples
        #     for sample in dataset['test']:
        #         if utterance == sample['audio']['path']:
        #             relevant_samples.append(sample)
        #             utterance_found = True
        #             break  # Found the relevant sample, move to the next utterance
        # if not utterance_found:
        #     # Iterate over the validation split to find relevant samples
        #     for sample in dataset['validation']:
        #         if utterance == sample['audio']['path']:
        #             relevant_samples.append(sample)
        #             utterance_found = True
        #             break  # Found the relevant sample, move to the next utterance
        # if utterance_found:
        #     break  # Found the utterance in the current dataset, move to the next utterance
    if not utterance_found:
        print(f"Utterance '{utterance}' not found in the dataset")
    else:
        print(f"Utterance '{utterance}' found in the dataset")

# If no samples were found, print an error message
if not relevant_samples:
    print("No relevant samples found. Exiting.")
    exit()

print(f"Found only {len(relevant_samples)} utterances in a dialogue of length {len(utterances_to_search)}")


Utterance 'Ses02M_impro05_M000.wav' found in the dataset

Utterance 'Ses02M_impro05_F000.wav' found in the dataset

Utterance 'Ses02M_impro05_M001.wav' found in the dataset

Utterance 'Ses02M_impro05_F001.wav' found in the dataset

Utterance 'Ses02M_impro05_M002.wav' found in the dataset

Utterance 'Ses02M_impro05_F002.wav' found in the dataset

Utterance 'Ses02M_impro05_F003.wav' found in the dataset

Utterance 'Ses02M_impro05_M003.wav' found in the dataset

Utterance 'Ses02M_impro05_F004.wav' found in the dataset

Utterance 'Ses02M_impro05_F005.wav' found in the dataset

Utterance 'Ses02M_impro05_F006.wav' found in the dataset

Utterance 'Ses02M_impro05_F007.wav' found in the dataset

Utterance 'Ses02M_impro05_M006.wav' found in the dataset

Utterance 'Ses02M_impro05_F008.wav' found in the dataset

Utterance 'Ses02M_impro05_F009.wav' found in the dataset

Utterance 'Ses02M_impro05_M008.wav' found in the dataset

Utterance 'Ses02M_impro05_F010.wav' found in the dataset

Utterance 'Ses

In [21]:
# Iterate over relevant samples to get ASR for each utterance
video_sentences= []
for idx, sample in enumerate(relevant_samples, start=1):
    audio_sample = sample['audio']
    waveform = audio_sample["array"]
    sampling_rate = audio_sample["sampling_rate"]

    # Use the model and processor to transcribe the audio:
    input_features = processor(
        waveform, sampling_rate=sampling_rate, return_tensors="pt"
    ).input_features

    # Generate token ids
    predicted_ids = model.generate(input_features)

    # Decode token ids to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    video_sentences.append(transcription[0])
    # print(f"Utterance {idx} ASR:")
    # print(transcription[0])
    # print("-" * 50)
print(video_sentences)

[' Hi', ' Hi, can I help you?', " It seems that my bag didn't make it out on the carousel.", ' Oh, did you check? Did you make sure that as it went around in second time?', " Yeah, yeah, it's gone around like you know 20 times it's been about half hour", ' Okay, well, um, the', " Actually, I'm looking here at the information. The slide that you were on, we had some trouble with some of the baggage. It looks like a few people's bags didn't make it through.", ' So what do I have to do to get', ' Well, you can fill out this paperwork and if for some reason it shows up, we can bring it to you. But...', " Well, sometimes it's just late baggage or misplaced baggage and it comes out, but it seems to be here and just that I'm reading off of this that", " It didn't make it into the second plane, so it looks like they don't actually know what happened to it.", " Well, sometimes there's difficulties with loading onto the plane. There can be a theft or some sort of mishap with a cart or something.

151 <class 'dict'> dict_keys(['Ses02F_script01_3', 'Ses03F_script03_1', 'Ses01F_script02_2', 'Ses02F_impro05', 'Ses05M_impro02', 'Ses04M_impro04', 'Ses02F_script01_1', 'Ses03M_impro08b', 'Ses05F_script01_2', 'Ses01F_impro03', 'Ses02M_script02_2', 'Ses03M_impro05b', 'Ses04F_impro07', 'Ses05M_script01_2', 'Ses01M_impro05', 'Ses03F_impro08', 'Ses02M_impro01', 'Ses05F_script03_2', 'Ses02M_impro05', 'Ses05M_script03_2', 'Ses05M_impro05', 'Ses04M_impro06', 'Ses04M_script03_2', 'Ses01F_impro04', 'Ses01F_script03_1', 'Ses01M_impro01', 'Ses03M_impro07', 'Ses05M_script01_3', 'Ses04F_script01_1', 'Ses03F_script02_1', 'Ses01M_impro03', 'Ses03M_script02_1', 'Ses01M_script01_3', 'Ses05F_impro03', 'Ses01M_impro02', 'Ses03M_script02_2', 'Ses05F_script03_1', 'Ses01M_script01_1', 'Ses03F_script03_2', 'Ses05F_script01_3', 'Ses03F_impro02', 'Ses03M_impro04', 'Ses03M_impro03', 'Ses05M_script02_1', 'Ses01M_script03_1', 'Ses04M_script03_1', 'Ses03M_script01_2', 'Ses01M_impro07', 'Ses03F_impro01', 'Ses03F_scr

In [ ]:
# List of utterances to search for in audio paths
utterances_to_search = [
    'Ses02F_impro05_F000.wav', 'Ses02F_impro05_M000.wav', 'Ses02F_impro05_F001.wav', 'Ses02F_impro05_M001.wav',
    'Ses02F_impro05_M002.wav', 'Ses02F_impro05_M003.wav', 'Ses02F_impro05_M004.wav', 'Ses02F_impro05_F004.wav',
    'Ses02F_impro05_M005.wav', 'Ses02F_impro05_M006.wav', 'Ses02F_impro05_F006.wav', 'Ses02F_impro05_M007.wav',
    'Ses02F_impro05_F007.wav', 'Ses02F_impro05_M008.wav', 'Ses02F_impro05_M009.wav', 'Ses02F_impro05_F010.wav',
    'Ses02F_impro05_M010.wav', 'Ses02F_impro05_F011.wav', 'Ses02F_impro05_M011.wav', 'Ses02F_impro05_M012.wav',
    'Ses02F_impro05_F013.wav', 'Ses02F_impro05_M013.wav', 'Ses02F_impro05_F014.wav', 'Ses02F_impro05_M014.wav',
    'Ses02F_impro05_F015.wav', 'Ses02F_impro05_M015.wav', 'Ses02F_impro05_F016.wav', 'Ses02F_impro05_M016.wav',
    'Ses02F_impro05_F017.wav', 'Ses02F_impro05_M017.wav', 'Ses02F_impro05_M018.wav', 'Ses02F_impro05_F018.wav',
    'Ses02F_impro05_M019.wav', 'Ses02F_impro05_F019.wav', 'Ses02F_impro05_M020.wav', 'Ses02F_impro05_F020.wav',
    'Ses02F_impro05_M021.wav', 'Ses02F_impro05_F022.wav'
]

# Initialize a list to store relevant samples
relevant_samples = []

# Iterate over each utterance to search for
for utterance in utterances_to_search:
    # Flag to indicate if the utterance is found
    utterance_found = False
    # Iterate over each dataset in all_datasets
    for dataset in all_datasets:
        # Iterate over the train split to find relevant samples
        for sample in dataset['train']:
            if utterance == sample['audio']['path']:
                relevant_samples.append(sample)
                utterance_found = True
                break  # Found the relevant sample, move to the next utterance
        if not utterance_found:
            # Iterate over the test split to find relevant samples
            for sample in dataset['test']:
                if utterance == sample['audio']['path']:
                    relevant_samples.append(sample)
                    utterance_found = True
                    break  # Found the relevant sample, move to the next utterance
        if not utterance_found:
            # Iterate over the validation split to find relevant samples
            for sample in dataset['validation']:
                if utterance == sample['audio']['path']:
                    relevant_samples.append(sample)
                    utterance_found = True
                    break  # Found the relevant sample, move to the next utterance
        if utterance_found:
            break  # Found the utterance in the current dataset, move to the next utterance
    if not utterance_found:
        print(f"Utterance '{utterance}' not found in the dataset")
    else:
        print(f"Utterance '{utterance}' found in the dataset")

# If no samples were found, print an error message
if not relevant_samples:
    print("No relevant samples found. Exiting.")
    exit()

# Iterate over relevant samples to get ASR for each utterance
for idx, sample in enumerate(relevant_samples, start=1):
    audio_sample = sample['audio']
    waveform = audio_sample["array"]
    sampling_rate = audio_sample["sampling_rate"]

    # Use the model and processor to transcribe the audio:
    input_features = processor(
        waveform, sampling_rate=sampling_rate, return_tensors="pt"
    ).input_features

    # Generate token ids
    predicted_ids = model.generate(input_features)

    # Decode token ids to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

    print(f"Utterance {idx} ASR:")
    print(transcription[0])
    print("-" * 50)

In [33]:
from datasets import load_dataset

dataset = load_dataset("confit/iemocap", "session1")


In [44]:
# Initialize an empty list to store datasets from each session
all_datasets = []

# Loop through sessions 1 to 5
for session_id in range(1, 6):
    # Load the dataset for the current session
    dataset = load_dataset("confit/iemocap", f"session{session_id}")
    # Append the dataset to the list of datasets
    all_datasets.append(dataset)

In [40]:
print(len(dataset['train']))
print(len(dataset['test']))
print(dataset['train'][2])
sorted_train = sorted(dataset['train'], key=lambda x: x['audio']['path'])

3556

1085

{'audio': {'path': 'Ses05M_script01_3_M020.wav', 'array': array([0.00320435, 0.00167847, 0.00057983, ..., 0.00204468, 0.0020752 ,

       0.00210571]), 'sampling_rate': 16000}, 'emotion': 'sad', 'label': 3}


In [43]:
print(sorted_train[2])

{'audio': {'path': 'Ses02F_impro01_F019.wav', 'array': array([-0.00152588, -0.0015564 , -0.00085449, ...,  0.00439453,

       -0.00125122, -0.00299072]), 'sampling_rate': 16000}, 'emotion': 'ang', 'label': 2}


In [4]:
dataset['train'][0]

{'audio': {'path': 'Ses01F_impro04_M010.wav',
  'array': array([-0.00619507, -0.00576782, -0.00546265, ...,  0.00964355,
          0.00973511,  0.0088501 ]),
  'sampling_rate': 16000},
 'emotion': 'neu',
 'label': 2}

In [27]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration


In [28]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.

  warnings.warn(


In [29]:
audio_sample = dataset['test'][0]['audio']
waveform = audio_sample["array"]
sampling_rate = audio_sample["sampling_rate"]

# Use the model and processor to transcribe the audio:
input_features = processor(
    waveform, sampling_rate=sampling_rate, return_tensors="pt"
).input_features

# Generate token ids
predicted_ids = model.generate(input_features)

# Decode token ids to text
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

transcription[0]

' I speak your language.'

Utterance 'Ses02F_impro05_F000.wav' found in the dataset

Utterance 'Ses02F_impro05_M000.wav' found in the dataset

Utterance 'Ses02F_impro05_F001.wav' found in the dataset

Utterance 'Ses02F_impro05_M001.wav' found in the dataset

Utterance 'Ses02F_impro05_M002.wav' found in the dataset

Utterance 'Ses02F_impro05_M003.wav' found in the dataset

Utterance 'Ses02F_impro05_M004.wav' found in the dataset

Utterance 'Ses02F_impro05_F004.wav' found in the dataset

Utterance 'Ses02F_impro05_M005.wav' found in the dataset

Utterance 'Ses02F_impro05_M006.wav' found in the dataset

Utterance 'Ses02F_impro05_F006.wav' found in the dataset

Utterance 'Ses02F_impro05_M007.wav' found in the dataset

Utterance 'Ses02F_impro05_F007.wav' not found in the dataset

Utterance 'Ses02F_impro05_M008.wav' found in the dataset

Utterance 'Ses02F_impro05_M009.wav' found in the dataset

Utterance 'Ses02F_impro05_F010.wav' found in the dataset

Utterance 'Ses02F_impro05_M010.wav' found in the dataset

Utterance 